In [1]:
# =========================
# IMPORTS
# =========================
import sqlite3
import csv
import urllib.request


# =========================
# FILE LINKS (from your GitHub repo)
# =========================
BASE_URL = "https://raw.githubusercontent.com/IsraSystems/CIS3120/main/"

BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"

BOOK_PATH   = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH   = "/content/Loan.csv"

DB_PATH = "/content/library.db"


# =========================
# DOWNLOAD FILES
# =========================
urllib.request.urlretrieve(BOOK_URL, BOOK_PATH)
urllib.request.urlretrieve(MEMBER_URL, MEMBER_PATH)
urllib.request.urlretrieve(LOAN_URL, LOAN_PATH)

print("Files downloaded!")


# =========================
# CREATE DATABASE
# =========================
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")


# =========================
# CREATE TABLES
# =========================
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT PRIMARY KEY,
    title TEXT,
    author TEXT
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER PRIMARY KEY,
    firstname TEXT,
    lastName TEXT
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT,
    id INTEGER,
    dateBorrowed TEXT,
    dateReturned TEXT,
    dateDue TEXT,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created!")


# =========================
# LOAD BOOK DATA
# =========================
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book VALUES (?, ?, ?)",
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
print("Books loaded:", conn.execute("SELECT COUNT(*) FROM Book").fetchone()[0])


# =========================
# LOAD MEMBER DATA
# =========================
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member VALUES (?, ?, ?)",
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print("Members loaded:", conn.execute("SELECT COUNT(*) FROM Member").fetchone()[0])


# =========================
# LOAD LOAN DATA
# =========================
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute(
            "INSERT INTO Loan VALUES (?, ?, ?, ?, ?)",
            (row['callNo'], int(row['id']), row['dateBorrowed'], returned, row['dateDue'])
        )

conn.commit()
print("Loans loaded:", conn.execute("SELECT COUNT(*) FROM Loan").fetchone()[0])


# =========================
# QUERY 1
# =========================
print("\nBooks sorted by author:")
for row in conn.execute("SELECT * FROM Book ORDER BY author"):
    print(row)


# =========================
# QUERY 2
# =========================
print("\nBooks not returned yet:")
for row in conn.execute("""
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL
"""):
    print(row)


# =========================
# QUERY 3
# =========================
print("\nLoan history for a book:")
for row in conn.execute("""
SELECT Member.firstname, Member.lastName, dateBorrowed, dateDue, dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY dateBorrowed
"""):
    print(row)


# =========================
# QUERY 4
# =========================
print("\nMembers with no loans:")
for row in conn.execute("""
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL
"""):
    print(row)


# =========================
# QUERY 5
# =========================
print("\nNumber of loans per member:")
for row in conn.execute("""
SELECT Member.firstname, Member.lastName, COUNT(Loan.callNo)
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id
ORDER BY COUNT(Loan.callNo) DESC
"""):
    print(row)


# =========================
# QUERY 6
# =========================
print("\nMost borrowed books:")
for row in conn.execute("""
SELECT Book.title, COUNT(*) as total
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
GROUP BY Book.title
ORDER BY total DESC
"""):
    print(row)


# =========================
# CLOSE DATABASE
# =========================
conn.close()

print("\nDone!")

Files downloaded!
Tables created!
Books loaded: 11
Members loaded: 4
Loans loaded: 4

Books sorted by author:
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')

Books not returned yet:
("Joe Celko's SQL puzzles & a

One thing I noticed is that some books have a NULL value for the dateReturned column, which means they haven’t been returned yet.  

A limitation of this dataset is that it is very small and doesn’t fully represent how a real library system works.  

Another limitation is that it doesn’t include important features like late fees, renewals, or book categories.  